# 1. Bidirection

## Weird Side-Effect

let's take a look over some visualizations of the skip patterns overlaide over the attention matrix $P = softmax(QK^T)$

[provide some images] \
**add explanation how to read the visualizations**

for some reason the left side get's much more skips then the right side...

## Why?

the iteration order inside LiteAttention is right to left and we in the pictures the max value for each row is roughly on the diagonal.\
as a result we find can't skip anything until we reach the diagonal.\
we will call this phenomenon "start range bias".

explain why global max is ideal.

## Possible Solutions (optional, not sure if it's needed)

1. Radial K-ordering - iterating from the diagonal, one to the right and one to the left. issue: heuristic (give an example for attention head where it's worse).
2. Max Location Save - saving the location of the tile with the max value and always start the iteration from it. issue: there is 128~ rows in each tile. should we save the max of each? if not it's another heuristic.

## Solution - Bidirectional Iteration

if we switch the iteration order between the time steps we would correct the "start range bias".\
[show an example over 3 consecutive timesteps]

## Implementation - Producer Consumer

while prototyping and exploring the different solutions it became tricky and error prone to change the iteration order\
in the Producer and Consumer side.\
to solve this problem we modified the way the consumer and producer agreed on the iteration order and switched to a full producer consumer pattern.

previously the producer and consumer had an independent loop which determiners the iteration order.
[show psudo code with producer iter loop and consumer iter loop]
but we want the producer to be able to tell the consumer what tile to run.
[show the new approch. sending the tile via shared memory]

Producer Implicit Padding

# LiteAttention - Low Level Optimization Tricks

in this blog we would mention a bunch of cool twiks and optimizations we added to LiteAttention.

## Succsessfull Optimizations

### Faster -INF padding

### Int To Float - Conversion Using Single Addition

### Instruction Mixing & Dependency Breaks

### Weird `warpgroup_wait` Optimization?
`shfl` between syncs??

## Failed Optimizations (but too cool to not mention)

### Swizzle During INT8 Quantization [optional, needs to verify further]

### Max Reduction & Exp2 Dependency Break!
using int max reduction while doing exp2 over the fractional parts.

### Int2Float Full Dependency Break!
setup the fragment with the magic constant. failed because the compiler feel very smart...